In [1]:
import torch
import torch.nn as nn

### Attention Block

In [11]:
class Attention(nn.Module):
    def __init__(self,features,num_heads):
        super().__init__()
        self.head_features=features//num_heads
        self.num_heads=num_heads
        self.q=nn.Linear(features,features)
        self.v=nn.Linear(features,features)
        self.k=nn.Linear(features,features)
    def forward(self,x):
        q=self.q(x)
        k=self.k(x)
        v=self.v(x)
        q=q.view(q.shape[0],q.shape[1],self.num_heads,self.head_features).transpose(1,2)
        k=k.view(k.shape[0],k.shape[1],self.num_heads,self.head_features).transpose(1,2)
        v=v.view(v.shape[0],v.shape[1],self.num_heads,self.head_features).transpose(1,2)
        score=q@k.transpose(-1,-2)/(self.head_features**0.5)
        mask=torch.tril(torch.ones(x.shape[1],x.shape[1]))
        score=score.masked_fill(mask==0,float("-inf"))
        score=score.softmax(dim=-1)
        attention=score@v
        attention=attention.contiguous().view(attention.shape[0],attention.shape[2],self.num_heads*self.head_features)
        return attention

### Masked Self Attention Execution

In [12]:
features=16
num_heads=4
x=torch.rand((2,20,16),dtype=torch.float32)
model=Attention(features,num_heads)
attention=model(x)
residue=attention+x
norm=nn.LayerNorm(features)
residue=norm(residue)
print(residue.shape)

torch.Size([2, 20, 16])


x.shape ==> [2,20,16]<br><br>
2 ==> number of batches/ sentences<br>
20==> number of tokens / words<br>
16==> features of each token<br>